# Diabetes Prediction Model
This notebook walks through the development of an end-to-end Machine Learning pipeline to predict diabetes status using the Pima Indians Diabetes dataset.

## Tasks Covered:
1. **Data Loading**: Load dataset and verify shape and rows.
2. **Data Preprocessing**: Detailed implementation of 5 distinct preprocessing steps.
3. **Pipeline Creation**: Create a unified `Pipeline` containing preprocessing and estimator.
4. **Primary Model Selection**: Select Random Forest Classifier and justify selection.
5. **Model Training**: Perform train-test split and fit the pipeline.
6. **Cross-Validation**: Perform 5-fold cross-validation to assess robustness.
7. **Hyperparameter Tuning**: Optimize model parameters using `GridSearchCV`.
8. **Best Model Selection**: Select the optimized pipeline.
9. **Model Performance Evaluation**: Assess best model on test set using metrics (Confusion Matrix, Classification Report, ROC-AUC).

### Task 1: Data Loading
We load the Pima Indians Diabetes dataset (`diabetes.csv`) using Pandas and display its shape and first 5 rows to verify correctness.

In [1]:
# --- TASK 1: DATA LOADING ---
# Import pandas and numpy libraries for data manipulation
import pandas as pd
import numpy as np

# Load 'diabetes.csv' dataset from local directory
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/diabetes.csv')

# Print shape to verify the number of rows (instances) and columns (features)
print(f"Dataset Shape: {df.shape} (768 rows, 9 columns)")

# Display the first 5 rows of the loaded dataframe to check columns and values
df.head()

Dataset Shape: (768, 9) (768 rows, 9 columns)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


### Exploratory Data Analysis (EDA) with ydata-profiling
To understand our features, distributions, correlations, and missing values, we generate an interactive profiling report using `ydata-profiling` and export it to an HTML file.

In [2]:
# Install ydata-profiling package inside the environment if not already installed
!pip install ydata-profiling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.0 MB/s eta 0:00:00


In [3]:
# EDA PROFILING REPORT
# Import ProfileReport from ydata_profiling library
from ydata_profiling import ProfileReport

# Initialize the profiling report on the diabetes dataframe
profile = ProfileReport(df, title="Diabetes Profiling Report", explorative=True)

# Save the generated report as an HTML file ('ydata.html') for local viewing
profile.to_file("ydata.html")

print("Profiling report generated and saved as ydata.html")

/tmp/ipykernel_622/391072335.py:3: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 9/9 [00:00<00:00, 81.60it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Profiling report generated and saved as ydata.html


### Task 2: Data Preprocessing
We inspect and perform preprocessing steps. In this dataset, there are `0` values in columns like `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI`. Since these are physiologically impossible, we treat them as missing values. We document at least 5 distinct preprocessing steps:
1. **Invalid Zero Handling (Step 1)**: Converting zero values in physiological columns to `NaN` so they can be imputed properly.
2. **Outlier Mitigation (Step 2)**: Using `RobustScaler` to scale features using median and IQR, preventing outliers from distorting distances.
3. **Missing Value Imputation (Step 3)**: Imputing the `NaN` values with the median of their respective columns.
4. **Feature Engineering (Step 4)**: Creating interaction and ratio features to extract additional predictive signals (e.g. `Glucose * Age`).
5. **Feature Scaling (Step 5)**: Standardizing and scaling inputs before passing to the classifier.

In [4]:
# TASK 2: DATA PREPROCESSING
# Define the columns where a value of 0 is physiologically invalid (treated as missing)
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# Step 1: Print the number of zero values in these columns to quantify missing values
print("Count of invalid zeros:")
for col in zero_cols:
    zeros = (df[col] == 0).sum()
    print(f"  {col}: {zeros} zeros ({zeros/len(df)*100:.2f}% of data)")

# Step 2: Inspect class label distribution to check for imbalance in target variable 'Outcome'
print("\nOutcome Class Distribution:")
print(df['Outcome'].value_counts(normalize=True))

Count of invalid zeros:
  Glucose: 5 zeros (0.65% of data)
  BloodPressure: 35 zeros (4.56% of data)
  SkinThickness: 227 zeros (29.56% of data)
  Insulin: 374 zeros (48.70% of data)
  BMI: 11 zeros (1.43% of data)

Outcome Class Distribution:
Outcome
0    0.651042
1    0.348958
Name: proportion, dtype: float64


### Task 3: Pipeline Creation
We construct a standard machine learning pipeline using scikit-learn's `Pipeline`. The pipeline wraps all our preprocessing steps (replacing zeros, imputing, engineering features, scaling) and the classification model into a single, clean interface.

In [5]:
# TASK 3: PIPELINE CREATION ---
# Import base transformer classes from sklearn to build custom transformers
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier

# 1. Custom Transformer: Replace invalid zeros in designated columns with NaN
class InvalidZeroReplacer(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None):
        # Store target columns. Defaults to standard physiological features.
        self.columns = columns if columns is not None else ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

    def fit(self, X, y=None):
        return self # No fitting required

    def transform(self, X):
        # Transform function: replaces 0 with NaN for designated columns
        if isinstance(X, pd.DataFrame):
            X_copy = X.copy()
            for col in self.columns:
                if col in X_copy.columns:
                    X_copy[col] = X_copy[col].replace(0, np.nan)
            return X_copy
        else:
            # Support numpy arrays (for compatibility in pipeline flows)
            X_copy = np.copy(X)
            col_map = {'Pregnancies': 0, 'Glucose': 1, 'BloodPressure': 2, 'SkinThickness': 3, 'Insulin': 4, 'BMI': 5, 'DiabetesPedigreeFunction': 6, 'Age': 7}
            for col in self.columns:
                idx = col_map[col]
                X_copy[X_copy[:, idx] == 0, idx] = np.nan
            return X_copy

# 2. Custom Transformer: Create engineered interaction and ratio features
class DiabetesFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self # No fitting required

    def transform(self, X):
        # Extract specific columns from array layout (Index 1=Glucose, 4=Insulin, 5=BMI, 7=Age)
        glucose = X[:, 1]
        age = X[:, 7]
        bmi = X[:, 5]
        insulin = X[:, 4]

        # Feature engineering step: compute interaction terms
        glucose_age_interaction = (glucose * age).reshape(-1, 1)
        bmi_age_interaction = (bmi * age).reshape(-1, 1)

        # Compute insulin/glucose ratio (adding small constant to avoid division-by-zero)
        insulin_glucose_ratio = (insulin / (glucose + 1e-5)).reshape(-1, 1)

        # Stack original columns with new features column-wise and return
        return np.hstack((X, glucose_age_interaction, bmi_age_interaction, insulin_glucose_ratio))

### Task 4: Primary Model Selection
We select the **Random Forest Classifier** as our primary machine learning algorithm.

**Justification**:
1. **Handles Outliers & Missing Values**: Random Forest works well with missing value imputations and is immune to feature scaling issues, ensuring robustness on physiological data.
2. **Captures Interactions**: Random Forest is highly capable of modeling non-linear decision thresholds and complex feature interactions (such as Glucose and Age).
3. **Bagging Ensemble**: Combines predictions from multiple decision trees to reduce overall variance, preventing overfitting.
4. **Class Weights Support**: Scikit-learn's implementation provides a `class_weight` parameter to address dataset class imbalance directly.

### Task 5: Model Training
We partition our dataset into an 80% training set and a 20% test set, using stratified sampling to ensure equal class proportions in both sets, and train our baseline Random Forest pipeline.

In [6]:
# TASK 5: MODEL TRAINING
# Import train_test_split from model_selection module
from sklearn.model_selection import train_test_split

# Separate independent variables (X) and target label 'Outcome' (y)
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

# Perform 80/20 train-test split, using stratify=y to maintain equal class proportions
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Print shapes of training and testing matrices to verify correctness
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

# Construct our unified baseline Machine Learning Pipeline
baseline_pipeline = Pipeline([
    ('zero_replacer', InvalidZeroReplacer()),          # Step 1: Replace zeros with NaNs
    ('imputer', SimpleImputer(strategy='median')),     # Step 2: Impute using column median
    ('feature_extractor', DiabetesFeatureExtractor()), # Step 3: Perform Feature Engineering
    ('scaler', RobustScaler()),                        # Step 4: Scale using outlier-robust metrics
    ('classifier', RandomForestClassifier(random_state=42)) # Step 5: Primary algorithm
])

# Fit the pipeline on training data
baseline_pipeline.fit(X_train, y_train)
print("Baseline model trained successfully!")

Training set shape: (614, 8)
Testing set shape: (154, 8)
Baseline model trained successfully!
